# Part 1 Data Cleaning and Validation

After analyzing all requirements, here is how I will structure the work:

Principles

- The raw table is never modified. Everything is read from ohlc_hw and written to new tables.
- Every raw row ends up in exactly one place: retained, removed (exact duplicate or all prices missing), or quarantined with a reason. 
- Repairs are flagged, never silent. Each repair gets a boolean flag column (e.g. flag_volume_filled, flag_ohlc_repaired, flag_adj_reconstructed, flag_extreme_return_kept).
- Original labels are preserved (raw_ticker), along with a ticker-normalization audit map.

Pipeline order (the order matters, since several steps depend on earlier ones)

- Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD. This happens before any grouping or deduplication.
- Profile (diagnostics only).Sort by (ticker, ts), and then
    - (low severity) exact dupli cate  
    - (high severity) conflicting key
    - (low severity) ticker format, ticker label that needed normalizing
    - ('medium' severity) Check missing cells per column, 
    - ('low' severity) missing trading dates versus the ohlc calendar (with a summary: "Ticker X is missing Y trading days between [start] and [end] dates"), 
    - ('medium' severity) impossible OHLC relationships, 
    - ('high' severity) non-positive prices, 
    - ('high' severity) flag extreme returns for manual checks, 
    - ('high' severity) flag zero volume days for manual checks.
- Generate a pandas dataframe 'Data Quality Report' based on the Profile (diagnostics only) step. The summary counts number of data issues per category. Column names are ticker | issue_type | issue_date | description | severity (high/medium/low).
- Deduplicate. Drop exact duplicates. For conflicting (ticker, ts) keys, resolve using evidence (the ohlc reference and OHLC validity), and quarantine the key if the evidence is inconclusive.
- Drop rows with all four raw prices missing, before any filling.
- - Sort by (ticker, ts), then repair raw prices. Invalid relationships become NaN, then forward-fill within ticker, then bounding with high = max(filled_high, open, close) and low = min(filled_low, open, close), with leading gaps quarantined. The OHLC bounds are rechecked afterwards, since a stale fill can violate today’s range. Leading gaps with nothing to carry forward are quarantined.
- Volume. Missing volume is filled with 0 and flagged.
- Extreme returns. Compute per-ticker z-scores as you specified based on repaired prices, then classify each hit:
    - Split-like: the volume jumps by a near-rational ratio
    - Data error: the move disagrees with the ohlc reference.
    - Inconclusive: quarantine and flag with a stated uncertainty.
- Only confirmed errors are interpolated, and only between valid neighbors.
- **Missing Adjusted close**. Take a date-specific factor adj_close / close from the matched ohlc row, then rebuild adj_close = clean_close × factor. There is no forward-filling of factors, and unresolved rows are quarantined and counted.

[*Note: the hw2.md document did not specify if vendor adj_close should be overwritten when they fall out of line with clean_close x factor. In the spirit of prudence, the design decision initially was to <u>NOT</u> overwrite, however, it produced charts that was unsmooth and unreadable. Therefore the hypothesis is that the initial requirement of hw2.md intended for adj_close to be overwriten but failed to mention it. Subsequently the code applied overwrites to these adj_close as if they were missing*]

- Final validation. Check unique keys, finite positive prices, finite nonnegative volume, OHLC bounds, and adjusted-price consistency.
- Show before/after stats. 
- Write outputs in data.db: ohlc_hw_cleaned, also ohlc_hw_quarantine (with reason), ohlc_hw_removed (with reason) and ohlc_hw_ticker_audit

## 1.1 Load and Inspect the Data

In [ ]:
# Load the `ohlc_hw` table from `data/data.db`

import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect("data/data.db")

df = pd.read_sql("SELECT * from ohlc_hw", conn)
df_calendar = pd.read_sql("SELECT * from ohlc", conn)
conn.close()

display(df.head())
display(df_calendar.head())

In [ ]:
# Display basic statistics (`.info()`, `.describe()`)

df.info()
df.describe()

**<u>My Observation</u>**

*Hypothesis: only adj_close has missing values*

**Identify the columns and their meanings**

This table contains daily OHLC (Open, High, Low, Close) data for multiple stocks.
The columns are ts (observation date), ticker, raw open, high, low, close, vendor adj_close, and volume


In [ ]:
# Show the date range and list of tickers
ts_col = pd.to_datetime(df['ts'])

#date range
print("date range in ohlc_hw")
ts_col.min(), ts_col.max()

In [ ]:
#compare with reference table ohlc
ts_col2 = pd.to_datetime(df_calendar['ts'])

#date range
print("date range in ohlc")
ts_col2.min(), ts_col2.max()

In [ ]:
#list of tickers (sorted)
ticker_col = df['ticker']
sorted(ticker_col.unique())

In [ ]:
# Load and normalize keys. Strip markup and whitespace and uppercase tickers, then parse ts to ISO YYYY-MM-DD.

def normalize_keys(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Normalize the (ticker, ts) key columns before any grouping or dedup.

    * ticker: remove HTML tags (e.g. '<a ...><span ...>SHW</span></a>' -> 'SHW'),
      strip whitespace, upper-case.
    * ts: parse to a date and re-emit as ISO 'YYYY-MM-DD' strings so SQL date
      filters and key comparisons behave consistently.
    """
    missing = {"ticker", "ts"} - set(raw.columns)
    if missing:
        raise KeyError(f"raw data is missing required columns: {sorted(missing)}")

    df = raw.copy()
    df["raw_ticker"], df["raw_ts"] = df["ticker"], df["ts"]

    df["ticker"] = (
        df["raw_ticker"].astype("string")
        .str.replace(r"<[^>]+>", "", regex=True)  # drop every HTML tag, keep inner text
        .str.strip()
        .str.upper()
        .mask(lambda s: s.eq(""))                 # label that was only markup -> missing
    )

    # errors="coerce" turns unparseable dates into NaT instead of aborting the load.
    df["ts"] = pd.to_datetime(df["raw_ts"], errors="coerce", format="mixed").dt.strftime("%Y-%m-%d")

    # The quarantine step reads this column to explain why a key is unusable.
    problems = pd.DataFrame({
        "unusable ticker": df["ticker"].isna(),
        "unparseable ts": df["ts"].isna(),
    })
    df["key_issue"] = problems.apply(lambda r: "; ".join(r.index[r]) or None, axis=1)

    ticker_audit = (
        df.groupby(["raw_ticker", "ticker"], dropna=False).size()
        .rename("n_rows").reset_index()
        .sort_values(["ticker", "raw_ticker"], na_position="last", ignore_index=True)
    )
    # A missing label counts as changed so it appears in the audit review.
    ticker_audit["was_changed"] = (
        ticker_audit["raw_ticker"].astype("string")
        .ne(ticker_audit["ticker"].astype("string"))
        .fillna(True)
        .astype(bool)
    )
    return df, ticker_audit

In [ ]:
# before and after ticker normalization
print("Unique tickers before normailization:", df["ticker"].nunique())
df, ticker_audit = normalize_keys(df)
print("Unique tickers after normalization: ", df["ticker"].nunique())

In [ ]:
# Missing in the raw columns vs. unusable after normalization
print("raw_ticker missing:", df["raw_ticker"].isna().sum())
print("raw_ts missing:    ", df["raw_ts"].isna().sum())
print("ticker missing after normalization:", df["ticker"].isna().sum())
print("ts missing after normalization:    ", df["ts"].isna().sum())

# Breakdown of rows with an unusable key, and the rows themselves
print(df["key_issue"].value_counts(dropna=False))
display(df[df["key_issue"].notna()].head())

## 1.2 Detect Data Quality Issues

In [ ]:
KEY = ["ticker", "ts"]
PRICE_COLS = ["open", "high", "low", "close"]
VALUE_COLS = PRICE_COLS + ["adj_close", "volume"]
REPORT_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]

# Single source of truth for severities, so every check and the summary agree.
SEVERITY = {
    "exact_duplicate": "low",
    "conflicting_key": "high",
    "ticker_format": "low",
    "missing_value": "medium",
    "missing_date": "low",
    "impossible_ohlc": "medium",
    "non_positive_price": "high",
    "extreme_return": "high",
    "zero_volume": "high",
}

df = df.sort_values(KEY, kind="stable", ignore_index=True)

# Rows with an unusable key cannot be located in the report, so checks skip them;
# the quarantine step deals with them later.
usable = df[df["key_issue"].isna()]
# Exact duplicates are dropped for diagnostics only (df is untouched), so one
# repeated record is not counted once per copy by the value checks below.
diag = usable.drop_duplicates(subset=KEY + VALUE_COLS)

dq_issues = {}  # issue_type -> report rows, assembled into the report in the last cell


def empty_report() -> pd.DataFrame:
    return pd.DataFrame(columns=REPORT_COLS)


def make_issues(ticker, issue_date, issue_type, description) -> pd.DataFrame:
    """Build Data Quality Report rows; scalar arguments broadcast over the Series ones."""
    out = pd.DataFrame({"ticker": ticker})
    out["issue_type"] = issue_type
    out["issue_date"] = issue_date
    out["description"] = description
    out["severity"] = SEVERITY[issue_type]
    return out[REPORT_COLS].reset_index(drop=True)


print(f"{len(df)} rows | {len(df) - len(usable)} with unusable key | "
      f"{len(usable) - len(diag)} exact duplicates | {len(diag)} rows used for value checks")

In [ ]:
def check_exact_duplicates(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag every extra copy of a row identical in key and all value columns."""
    # keep="first": the first occurrence is the record, later copies are the issue.
    dups = rows[rows.duplicated(KEY + VALUE_COLS, keep="first")]
    return make_issues(dups["ticker"], dups["ts"], "exact_duplicate",
                       "Identical to an earlier row (after ticker normalization)")


dq_issues["exact_duplicate"] = check_exact_duplicates(usable)
print(len(dq_issues["exact_duplicate"]), "exact duplicate rows")
display(dq_issues["exact_duplicate"].head())

In [ ]:
def check_conflicting_keys(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag (ticker, ts) keys that still hold different values once exact duplicates are gone.

    * One issue per key, naming the columns that disagree, so manual review knows where to look.
    * `rows` must already be free of exact duplicates.
    """
    conflicts = rows[rows.duplicated(KEY, keep=False)]
    if conflicts.empty:
        return empty_report()
    by_key = conflicts.groupby(KEY)
    n_rows = by_key.size().rename("n_rows")
    disagree = by_key[VALUE_COLS].nunique(dropna=False).gt(1)
    cols = disagree.apply(lambda r: ", ".join(r.index[r]), axis=1).rename("cols")
    out = pd.concat([n_rows, cols], axis=1).reset_index()
    desc = out["n_rows"].astype(str) + " rows share this key but differ in: " + out["cols"]
    return make_issues(out["ticker"], out["ts"], "conflicting_key", desc)


dq_issues["conflicting_key"] = check_conflicting_keys(diag)
print(len(dq_issues["conflicting_key"]), "conflicting keys")
display(dq_issues["conflicting_key"].head())

In [ ]:
def check_ticker_format(audit: pd.DataFrame) -> pd.DataFrame:
    """One issue per raw ticker label that needed normalizing (from the audit map)."""
    changed = audit[audit["was_changed"]]
    desc = pd.Series(
        [f"Label {raw!r} ({n} rows) " + (f"normalized to {tkr!r}" if pd.notna(tkr) else "left no usable ticker")
         for raw, tkr, n in zip(changed["raw_ticker"], changed["ticker"], changed["n_rows"])],
        index=changed.index, dtype=object,
    )
    # A label-level issue has no single date, so issue_date stays empty.
    return make_issues(changed["ticker"], None, "ticker_format", desc)


dq_issues["ticker_format"] = check_ticker_format(ticker_audit)
print(len(dq_issues["ticker_format"]), "labels needed normalizing")
display(dq_issues["ticker_format"].head())

In [ ]:
def check_missing_cells(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per missing cell in the price, adj_close and volume columns."""
    long = rows.melt(id_vars=KEY, value_vars=VALUE_COLS, var_name="column")
    missing = long[long["value"].isna()]
    return make_issues(missing["ticker"], missing["ts"], "missing_value", "Missing " + missing["column"])


print("Missing cells per column:")
display(diag[VALUE_COLS].isna().sum().to_frame("n_missing"))  # tests the "only adj_close is missing" hypothesis

dq_issues["missing_value"] = check_missing_cells(diag)
print(len(dq_issues["missing_value"]), "missing cells")

In [ ]:
def check_missing_dates(rows: pd.DataFrame, calendar_ts: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare each ticker's dates with the trading days in the ohlc reference table.

    * The window is the ticker's own first-to-last date in ohlc_hw, so a ticker that
      starts late or ends early is not charged for dates outside its history.
    * Returns (report rows, one summary row per ticker).
    """
    calendar = pd.Index(sorted(pd.to_datetime(calendar_ts).dt.strftime("%Y-%m-%d").dropna().unique()))
    frames = []
    for ticker, g in rows.groupby("ticker"):
        seen = pd.Index(g["ts"].unique())
        start, end = seen.min(), seen.max()
        in_window = calendar[(calendar >= start) & (calendar <= end)]
        missing = in_window.difference(seen)
        if len(missing):
            frames.append(pd.DataFrame({"ticker": ticker, "ts": missing, "start": start, "end": end}))
    if not frames:
        return empty_report(), pd.DataFrame(columns=["ticker", "n_missing", "start", "end"])

    gaps = pd.concat(frames, ignore_index=True)
    summary = (gaps.groupby("ticker")
               .agg(n_missing=("ts", "size"), start=("start", "first"), end=("end", "first"))
               .reset_index())
    report = make_issues(gaps["ticker"], gaps["ts"], "missing_date",
                         "Trading day in the ohlc calendar has no row in ohlc_hw")
    return report, summary


dq_issues["missing_date"], missing_date_summary = check_missing_dates(diag, df_calendar["ts"])

for r in missing_date_summary.itertuples():
    print(f"Ticker {r.ticker} is missing {r.n_missing} trading days between {r.start} and {r.end}")
print(len(dq_issues["missing_date"]), "missing ticker-dates in total")

In [ ]:
def check_impossible_ohlc(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag rows that break the OHLC relationships.

    * high must be >= max(open, close) and >= low; low must be <= min(open, close).
    * A NaN cell never violates a rule (it is reported as missing_value instead).
    """
    # skipna: if open is missing, close alone sets the bound.
    body_top = rows[["open", "close"]].max(axis=1)
    body_bottom = rows[["open", "close"]].min(axis=1)
    rules = pd.DataFrame({
        "high < max(open, close)": rows["high"] < body_top,
        "low > min(open, close)": rows["low"] > body_bottom,
        "high < low": rows["high"] < rows["low"],
    })
    bad = rules.any(axis=1)
    if not bad.any():
        return empty_report()
    desc = rules[bad].apply(lambda r: "; ".join(r.index[r]), axis=1)  # which rules failed
    return make_issues(rows.loc[bad, "ticker"], rows.loc[bad, "ts"], "impossible_ohlc", desc)


dq_issues["impossible_ohlc"] = check_impossible_ohlc(diag)
print(len(dq_issues["impossible_ohlc"]), "rows with impossible OHLC relationships")
display(dq_issues["impossible_ohlc"].head())

In [ ]:
def check_non_positive_prices(rows: pd.DataFrame) -> pd.DataFrame:
    """One issue per open/high/low/close/adj_close cell that is <= 0 or infinite."""
    long = rows.melt(id_vars=KEY, value_vars=PRICE_COLS + ["adj_close"], var_name="column")
    bad = long[long["value"].le(0) | np.isinf(long["value"])]  # NaN is not flagged here
    desc = "Non-positive/non-finite " + bad["column"] + " = " + bad["value"].astype(str)
    return make_issues(bad["ticker"], bad["ts"], "non_positive_price", desc)


dq_issues["non_positive_price"] = check_non_positive_prices(diag)
print(len(dq_issues["non_positive_price"]), "non-positive price cells")
display(dq_issues["non_positive_price"].head())

In [ ]:
def check_extreme_returns(rows: pd.DataFrame, n_std: float = 3) -> pd.DataFrame:
    """Flag closes whose daily return is more than n_std std away from the ticker's mean return.

    * Rule: abs(ret - mean_ret) > n_std * std_ret per ticker (sample std, ddof=1).
    * Returns skip missing and non-positive closes (a return spans the gap) and
      unresolved conflicting keys, so one bad row cannot poison the chain.
    * Diagnostic only: the cleaning step recomputes this on repaired prices.
    """
    clean = ~rows.duplicated(KEY, keep=False) & rows["close"].gt(0)
    s = rows.loc[clean, KEY + ["close"]].sort_values(KEY)

    s["prev_ts"] = s.groupby("ticker")["ts"].shift()
    s["ret"] = s["close"] / s.groupby("ticker")["close"].shift() - 1
    s["dev"] = s["ret"] - s.groupby("ticker")["ret"].transform("mean")
    s["std"] = s.groupby("ticker")["ret"].transform("std")

    hits = s[s["dev"].abs() > n_std * s["std"]]  # NaN std (too few returns) never flags
    if hits.empty:
        return empty_report()
    desc = hits.apply(
        lambda r: f"close {r['prev_ts']} -> {r['ts']}: return {r['ret']:+.1%} "
                  f"({r['dev'] / r['std']:+.1f} std from ticker mean)", axis=1)
    return make_issues(hits["ticker"], hits["ts"], "extreme_return", desc)


dq_issues["extreme_return"] = check_extreme_returns(diag)
print(len(dq_issues["extreme_return"]), "extreme returns to review")
display(dq_issues["extreme_return"].head())

In [ ]:
def check_zero_volume(rows: pd.DataFrame) -> pd.DataFrame:
    """Flag days with volume == 0. Missing volume is a separate issue (missing_value)."""
    zero = rows[rows["volume"].eq(0)]
    return make_issues(zero["ticker"], zero["ts"], "zero_volume", "Volume is zero: review manually")


dq_issues["zero_volume"] = check_zero_volume(diag)
print(len(dq_issues["zero_volume"]), "zero-volume days")
display(dq_issues["zero_volume"].head())

In [ ]:
# Data Quality Report and summary

SEVERITY_ORDER = ["high", "medium", "low"]

frames = [f for f in dq_issues.values() if not f.empty]
dq_report = pd.concat(frames, ignore_index=True)
dq_report["severity"] = pd.Categorical(dq_report["severity"], categories=SEVERITY_ORDER, ordered=True)
dq_report = dq_report.sort_values(["severity", "issue_type", "ticker", "issue_date"], ignore_index=True)

# Summary lists every check, including those with zero hits, so it is clear each one ran.
dq_summary = (
    pd.DataFrame({
        "issue_type": list(dq_issues),
        "severity": [SEVERITY[k] for k in dq_issues],
        "n_issues": [len(v) for v in dq_issues.values()],
    })
    .assign(severity=lambda d: pd.Categorical(d["severity"], categories=SEVERITY_ORDER, ordered=True))
    .sort_values(["severity", "n_issues"], ascending=[True, False], ignore_index=True)
)

display(dq_summary)
print("Total issues:", len(dq_report))
display(dq_report.head(20))

In [ ]:
print(list(df.columns))

## 1.3 Clean the Data

In [ ]:
## Deduplicate

REF_RTOL = 1e-3  # relative tolerance when comparing a price with the ohlc reference (tunable)

# Stable id so every raw row ends up in exactly one place: retained, removed or quarantined.
# NOTE: df was sorted in 1.2, so this id is the position in that stable sort. Load with
# "SELECT rowid AS raw_row_id, * FROM ohlc_hw" if you want the table's own rowid instead.
if "raw_row_id" not in df.columns:
    df.insert(0, "raw_row_id", np.arange(len(df)))
N_RAW = len(df)

LOG_COLS = ["raw_row_id", "raw_ticker", "raw_ts", "ticker", "ts"] + VALUE_COLS
removed_log, quarantine_log = [], []


def log_rows(log: list, rows: pd.DataFrame, reason: str, note=None) -> None:
    """Record rows leaving the pipeline together with the reason (and an optional note)."""
    if rows.empty:
        return
    out = rows[LOG_COLS].copy()
    out["reason"], out["note"] = reason, note
    log.append(out)


def collect(log: list) -> pd.DataFrame:
    return pd.concat(log, ignore_index=True) if log else pd.DataFrame(columns=LOG_COLS + ["reason", "note"])


def ohlc_violations(rows: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    """Boolean masks of rows whose high / low break the OHLC relationships.

    * high is bad if high < max(open, close) or high < low; low is bad if
      low > min(open, close) or low > high. When high < low both are bad.
    * NaN never violates a rule (missing cells are handled separately).
    """
    top, bottom = rows[["open", "close"]].max(axis=1), rows[["open", "close"]].min(axis=1)
    high_bad = (rows["high"] < top) | (rows["high"] < rows["low"])
    low_bad = (rows["low"] > bottom) | (rows["low"] > rows["high"])
    return high_bad, low_bad


def bound_ohlc(prices: pd.DataFrame) -> pd.DataFrame:
    """Widen high/low so the row's own open and close always sit inside [low, high]."""
    out = prices.copy()
    out["high"] = prices[["high", "open", "close"]].max(axis=1)
    out["low"] = prices[["low", "open", "close"]].min(axis=1)
    return out


def assert_price_integrity(rows: pd.DataFrame, step: str) -> None:
    """Recheck after every repair: finite positive prices and valid OHLC bounds."""
    prices = rows[PRICE_COLS]
    assert np.isfinite(prices).all().all() and (prices > 0).all().all(), f"{step}: missing/non-positive price"
    high_bad, low_bad = ohlc_violations(rows)
    assert not (high_bad | low_bad).any(), f"{step}: OHLC bounds violated"


# Reference table: same normalization, one trusted row per (ticker, ts).
ref, _ = normalize_keys(df_calendar)
ref = ref[ref["key_issue"].isna()].drop_duplicates(KEY + VALUE_COLS)
ref = ref[~ref.duplicated(KEY, keep=False)]  # conflicting reference keys are not trustworthy evidence
ref_cols = PRICE_COLS + ["adj_close"]
ref = ref[KEY + ref_cols].rename(columns={c: f"ref_{c}" for c in ref_cols})
# Date-specific adjustment factor; defined only where both reference prices are finite and positive.
pair = ref[["ref_close", "ref_adj_close"]]
ref["ref_factor"] = (pair["ref_adj_close"] / pair["ref_close"]).where(np.isfinite(pair).all(axis=1) & (pair > 0).all(axis=1))


def resolve_conflicting_keys(rows: pd.DataFrame, ref: pd.DataFrame) -> tuple[list, list]:
    """Return (ids to remove, ids to quarantine) for (ticker, ts) keys holding different values.

    * Evidence 1: the row's open/high/low/close all match the ohlc reference.
    * Evidence 2: the row has all prices present and no OHLC violation.
    * Evidence 1 is tried first; a key is resolved only if exactly one row wins on that
      evidence, otherwise every row of the key is quarantined.
    """
    conf = rows[rows.duplicated(KEY, keep=False)].merge(ref, on=KEY, how="left")
    conf["matches_ref"] = np.column_stack(
        [np.isclose(conf[c], conf[f"ref_{c}"], rtol=REF_RTOL) for c in PRICE_COLS]).all(axis=1)
    high_bad, low_bad = ohlc_violations(conf)
    conf["is_valid"] = ~(high_bad | low_bad) & conf[PRICE_COLS].notna().all(axis=1)

    drop, quarantine = [], []
    for _, g in conf.groupby(KEY):
        winner = next((g.loc[g[c], "raw_row_id"].iloc[0] for c in ("matches_ref", "is_valid") if g[c].sum() == 1), None)
        if winner is None:
            quarantine += g["raw_row_id"].tolist()
        else:
            drop += [i for i in g["raw_row_id"] if i != winner]
    return drop, quarantine


work = df.copy()

# 1) Unusable keys cannot be deduplicated or located: quarantine with the reason from normalize_keys.
bad_key = work["key_issue"].notna()
log_rows(quarantine_log, work[bad_key], "unusable key", work.loc[bad_key, "key_issue"])
work = work[~bad_key]

# 2) Exact duplicates (same key and identical values, NaN == NaN): keep the first copy.
exact_dup = work.duplicated(KEY + VALUE_COLS, keep="first")
log_rows(removed_log, work[exact_dup], "exact duplicate")
work = work[~exact_dup]

# 3) Conflicting keys: resolve with evidence, quarantine when inconclusive.
drop_ids, quarantine_ids = resolve_conflicting_keys(work, ref)
log_rows(removed_log, work[work["raw_row_id"].isin(drop_ids)], "conflicting key: lost to evidence")
log_rows(quarantine_log, work[work["raw_row_id"].isin(quarantine_ids)], "conflicting key: inconclusive evidence")
work = work[~work["raw_row_id"].isin(drop_ids + quarantine_ids)]
assert not work.duplicated(KEY).any(), "(ticker, ts) is still not unique"

print(f"raw {N_RAW} | unusable key {bad_key.sum()} | exact duplicates {exact_dup.sum()} | "
      f"conflicts removed {len(drop_ids)}, quarantined {len(quarantine_ids)} | left {len(work)}")

In [ ]:
## Drop rows with all four raw prices missing (before any filling)

# A row with no price at all carries nothing to repair; filling it would invent a whole bar.
all_missing = work[PRICE_COLS].isna().all(axis=1)
log_rows(removed_log, work[all_missing], "all raw prices missing")
work = work[~all_missing]
print(f"removed {all_missing.sum()} rows with no raw price | left {len(work)}")

In [ ]:
## Repair raw prices

def repair_raw_prices(rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repair open/high/low/close within each ticker and flag every change.

    * Non-positive / non-finite prices are treated like missing cells.
    * Impossible OHLC: high -> NaN and/or low -> NaN (open and close are never touched).
    * Forward-fill within ticker, then bound: high = max(high, open, close), low = min(low, open, close).
    * Rows still missing a price after the fill (leading gap) are returned separately for quarantine.
    """
    rows = rows.sort_values(KEY, kind="stable").copy()  # fill and shift need (ticker, ts) order

    prices = rows[PRICE_COLS].replace([np.inf, -np.inf], np.nan)
    prices = prices.where(prices > 0)  # non-positive -> NaN
    was_missing = prices.isna()

    high_bad, low_bad = ohlc_violations(prices)
    prices.loc[high_bad, "high"] = np.nan
    prices.loc[low_bad, "low"] = np.nan

    # groupby(ticker).ffill never borrows a value across tickers.
    filled = prices.groupby(rows["ticker"]).ffill()
    leading = filled.isna().any(axis=1)  # nothing earlier in the ticker to carry forward

    # Bound only rows without a gap: bounding a NaN high would silently invent max(open, close).
    bounded = filled.copy()
    bounded.loc[~leading] = bound_ohlc(filled.loc[~leading])
    # A stale fill can fall outside today's range; this catches rows the bound had to widen.
    bound_changed = (bounded[["high", "low"]] != filled[["high", "low"]]).any(axis=1) & ~leading

    rows[PRICE_COLS] = bounded
    rows["flag_price_filled"] = was_missing.any(axis=1)
    rows["flag_close_repaired"] = was_missing["close"]  # close is never changed by bounding; used for adj_close later
    rows["flag_ohlc_repaired"] = high_bad | low_bad | bound_changed

    gap_cols = [", ".join(c for c, m in zip(PRICE_COLS, flags) if m) for flags in filled[leading].isna().to_numpy()]
    leading_rows = rows[leading].assign(note=[f"no prior value for: {c}" for c in gap_cols])
    return rows[~leading], leading_rows


work, leading_rows = repair_raw_prices(work)
log_rows(quarantine_log, leading_rows, "leading gap", leading_rows["note"])
assert_price_integrity(work, "after raw-price repair")


print(f"quarantined {len(leading_rows)} leading-gap rows | left {len(work)}")
display(work[["flag_price_filled", "flag_close_repaired", "flag_ohlc_repaired"]].sum().to_frame("n_rows"))

In [ ]:
## Volume

# Unknown volume is not evidence of no trading, so the fill value (0) is always paired with a flag.
# Negative or infinite volume cannot be real either; it is handled the same way.
bad_volume = ~np.isfinite(work["volume"]) | work["volume"].lt(0)
work["flag_volume_filled"] = bad_volume
work.loc[bad_volume, "volume"] = 0.0
print(f"filled {bad_volume.sum()} volume cells with 0 (flagged)")

In [ ]:
## Extreme returns: classify each hit

N_STD = 3
# Volume ratios treated as split-like (2-for-1, 3-for-2, reverse splits ...) and their tolerance (tunable).
SPLIT_RATIOS = np.array([2, 3, 4, 5, 10, 1/2, 1/3, 1/4, 1/5, 1/10, 3/2, 2/3])
SPLIT_TOL = 0.02
KEPT_CLASSES = ["split_like", "ref_confirmed", "revert_of_data_error"]


def zscore_returns(rows: pd.DataFrame, n_std: float = N_STD) -> pd.DataFrame:
    """Daily close return per ticker and whether abs(ret - mean_ret) > n_std * std_ret.

    * Sample std (ddof=1), per ticker, on the sorted repaired closes.
    * A ticker with std 0 or fewer than 2 returns never flags.
    """
    s = rows.sort_values(KEY, kind="stable").reset_index(drop=True)
    by_ticker = s.groupby("ticker")
    s["prev_close"] = by_ticker["close"].shift()
    s["prev_volume"] = by_ticker["volume"].shift()
    s["prev_vol_filled"] = by_ticker["flag_volume_filled"].shift(fill_value=True)
    s["ret"] = s["close"] / s["prev_close"] - 1
    stats = s.groupby("ticker")["ret"]
    # |ret - mean| > n * std  <=>  |z| > n
    s["zscore"] = (s["ret"] - stats.transform("mean")) / stats.transform("std")
    s["is_hit"] = s["zscore"].abs() > n_std
    return s


def classify_extreme_returns(s: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    """Label each hit with the evidence available.

    * data_error: close disagrees with the ohlc reference (checked first; the reference wins over volume).
    * split_like: volume jumps by a near-rational ratio (needs real volume on both days).
    * ref_confirmed: close agrees with the reference, so the move is real news: keep and flag.
    * revert_of_data_error: otherwise-inconclusive hit right after a data error (it is the reversion).
    * inconclusive: no evidence either way -> quarantined.
    """
    s = s.merge(ref[KEY + ["ref_close"]], on=KEY, how="left")  # left merge keeps the sorted order
    ref_agrees = pd.Series(np.isclose(s["close"], s["ref_close"], rtol=REF_RTOL), index=s.index)  # False if no ref
    ref_disagrees = s["ref_close"].notna() & ~ref_agrees

    # Volume ratio is undefined for zero, missing or filled volume: such hits cannot be called splits.
    vol_known = (s["volume"] > 0) & (s["prev_volume"] > 0) & ~s["flag_volume_filled"] & ~s["prev_vol_filled"]
    s["vol_ratio"] = (s["volume"] / s["prev_volume"]).where(vol_known)
    s["close_ratio"] = 1 + s["ret"]  # evidence only: a split should show close_ratio ~ 1 / vol_ratio
    s["close_vs_ref"] = s["close"] / s["ref_close"] - 1
    is_split = np.abs(s["vol_ratio"].to_numpy()[:, None] / SPLIT_RATIOS - 1).min(axis=1) <= SPLIT_TOL

    outcome = np.select([ref_disagrees, is_split, ref_agrees],
                        ["data_error", "split_like", "ref_confirmed"], default="inconclusive")
    s["er_class"] = np.where(s["is_hit"], outcome, None)

    prev_class = s.groupby("ticker")["er_class"].shift()
    s.loc[s["er_class"].eq("inconclusive") & prev_class.eq("data_error"), "er_class"] = "revert_of_data_error"

    # Stated uncertainty for the quarantine table.
    vr = s["vol_ratio"].round(3).astype(str)
    s["note"] = np.where(s["vol_ratio"].isna(),
                         "No ohlc reference row; volume ratio undefined (zero, missing or filled volume)",
                         "No ohlc reference row; volume ratio " + vr + " is not a near-rational split ratio")
    return s


er = classify_extreme_returns(zscore_returns(work), ref)
er_hits = er[er["is_hit"]]
display(er_hits["er_class"].value_counts().to_frame("n_hits"))
display(er_hits[["ticker", "ts", "ret", "zscore", "vol_ratio", "close_ratio", "close_vs_ref", "er_class"]].head(10))

# Attach the class to the working table; the z-scores above were computed before any row is dropped.
by_id = er_hits.set_index("raw_row_id")
work["extreme_return_class"] = work["raw_row_id"].map(by_id["er_class"])
work["flag_extreme_return_kept"] = work["extreme_return_class"].isin(KEPT_CLASSES)

# Inconclusive: quarantine with the stated uncertainty.
inconclusive = work["extreme_return_class"].eq("inconclusive")
log_rows(quarantine_log, work[inconclusive], "inconclusive extreme return", work.loc[inconclusive, "raw_row_id"].map(by_id["note"]))
work = work[~inconclusive]
print(f"quarantined {inconclusive.sum()} inconclusive rows | left {len(work)}")

In [ ]:
## Interpolate confirmed data errors (between valid neighbors only)

def interpolate_data_errors(rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Replace the prices of confirmed data-error rows by linear interpolation within ticker.

    * All four prices of the row are rebuilt (a bad close makes the rest of the bar unreliable),
      then re-bounded so open/close sit inside [low, high].
    * limit_area="inside": never extrapolate. Errors without a valid neighbor on both sides
      are returned separately for quarantine.
    * Only error rows are NaN here, so no other cell can be interpolated by accident.
    """
    rows = rows.sort_values(KEY, kind="stable").copy()
    is_error = rows["extreme_return_class"].eq("data_error")

    prices = rows[PRICE_COLS].copy()
    prices.loc[is_error] = np.nan
    interp = prices.groupby(rows["ticker"]).transform(lambda s: s.interpolate(limit_area="inside"))  # linear by position

    unresolved = is_error & interp.isna().any(axis=1)
    fixed = is_error & ~unresolved

    bounded = bound_ohlc(interp.loc[fixed])
    bound_changed = (bounded[["high", "low"]] != interp.loc[fixed, ["high", "low"]]).any(axis=1)

    rows.loc[fixed, PRICE_COLS] = bounded
    rows["flag_extreme_return_interpolated"] = fixed
    rows["flag_close_repaired"] |= fixed          # adj_close must be rebuilt for these rows
    rows.loc[fixed[fixed].index[bound_changed], "flag_ohlc_repaired"] = True
    return rows[~unresolved], rows[unresolved]


work, unresolved_errors = interpolate_data_errors(work)
log_rows(quarantine_log, unresolved_errors, "confirmed data error without valid neighbors",
         "close disagrees with ohlc reference; cannot interpolate without a valid neighbor on both sides")
assert_price_integrity(work, "after interpolation")

print(f"interpolated {work['flag_extreme_return_interpolated'].sum()} rows | "
      f"quarantined {len(unresolved_errors)} | left {len(work)}")
# Recheck on the repaired series (statistics change once the errors are gone).
print("extreme returns remaining on repaired closes:", zscore_returns(work)["is_hit"].sum())

In [ ]:
"""
## Missing adjusted close (initial design)

ADJ_RTOL = 1e-3  # tolerance for the consistency diagnostic below

# Convention: adj_close = clean_close * (reference adj_close / reference close) on the SAME date.
# The factor is never carried across dates, so a corporate action cannot leak into other days.
work = work.merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")

adj_invalid = ~np.isfinite(work["adj_close"]) | work["adj_close"].le(0)
# A vendor adj_close no longer matches a close that was filled or interpolated, so rebuild those too.
rebuild = adj_invalid | work["flag_close_repaired"]
has_factor = work["ref_factor"].notna()

unresolved = rebuild & ~has_factor
note = np.where(adj_invalid, "adj_close missing/invalid and no trusted reference factor",
                "close was repaired and no trusted reference factor")
log_rows(quarantine_log, work[unresolved], "adj_close unresolved", pd.Series(note, index=work.index)[unresolved])

work["flag_adj_reconstructed"] = rebuild & has_factor
work.loc[work["flag_adj_reconstructed"], "adj_close"] = work["close"] * work["ref_factor"]
work = work[~unresolved]

# Consistency diagnostic on untouched vendor rows: does adj_close / close match the reference factor?
vendor = work[~work["flag_adj_reconstructed"] & work["ref_factor"].notna()]
mismatch = ~np.isclose(vendor["adj_close"] / vendor["close"], vendor["ref_factor"], rtol=ADJ_RTOL)
print(f"adj_close rebuilt {work['flag_adj_reconstructed'].sum()} | quarantined {unresolved.sum()} | "
      f"vendor rows disagreeing with reference factor: {mismatch.sum()} of {len(vendor)} (reported, not altered)")

work = work.drop(columns="ref_factor").reset_index(drop=True)
assert_price_integrity(work, "after adj_close rebuild")
assert np.isfinite(work["adj_close"]).all() and (work["adj_close"] > 0).all()
"""


In [ ]:
## Missing adjusted close (variant: also rebuild vendor adj_close that disagrees with close x factor)

ADJ_RTOL = 1e-3  # relative tolerance for "vendor adj_close / close agrees with the reference factor"


def rebuild_adj_close(rows: pd.DataFrame, ref: pd.DataFrame, reconstruct_inconsistent: bool):
    """Rebuild adj_close = clean close x same-date reference factor (ohlc adj_close / close).

    * Always rebuilt: adj_close missing/invalid, or close repaired earlier (filled/interpolated).
    * Only if `reconstruct_inconsistent`: vendor adj_close / close differs from the same-date
      reference factor by more than ADJ_RTOL. False reproduces the original cell exactly.
    * The factor is never carried across dates. Rows that need a rebuild but have no reference
      factor are returned for quarantine; vendor rows with no factor cannot be checked and are kept.
    * Pure: the caller does the logging, so the function can be re-run for comparisons.

    Returns (kept rows incl. ref_factor, unresolved rows, quarantine note per unresolved row).
    """
    rows = rows.merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")
    has_factor = rows["ref_factor"].notna()

    adj_invalid = ~np.isfinite(rows["adj_close"]) | rows["adj_close"].le(0)
    close_repaired = rows["flag_close_repaired"]  # vendor adj_close no longer matches a repaired close

    # Vendor-only inconsistency: a valid adj_close on an untouched close that contradicts the reference.
    agrees = pd.Series(np.isclose(rows["adj_close"] / rows["close"], rows["ref_factor"], rtol=ADJ_RTOL),
                       index=rows.index)
    inconsistent = has_factor & ~adj_invalid & ~close_repaired & ~agrees

    rebuild = adj_invalid | close_repaired | (inconsistent if reconstruct_inconsistent else False)
    unresolved = (adj_invalid | close_repaired) & ~has_factor  # inconsistent rows always have a factor

    note = pd.Series(np.where(adj_invalid, "adj_close missing/invalid and no trusted reference factor",
                              "close was repaired and no trusted reference factor"), index=rows.index)

    rows["flag_adj_vendor_inconsistent"] = inconsistent & reconstruct_inconsistent  # subset of flag_adj_reconstructed
    rows["flag_adj_reconstructed"] = rebuild & has_factor
    rows.loc[rows["flag_adj_reconstructed"], "adj_close"] = rows["close"] * rows["ref_factor"]
    return rows[~unresolved], rows[unresolved], note[unresolved]


work_pre_adj = work.copy()  # snapshot for the comparison test; re-run from the top, this cell is not idempotent
work, unresolved, note = rebuild_adj_close(work, ref, reconstruct_inconsistent=True)
log_rows(quarantine_log, unresolved, "adj_close unresolved", note)

# Self-check: after the rebuild, no kept vendor value may contradict the reference factor.
kept_vendor = work[~work["flag_adj_reconstructed"] & work["ref_factor"].notna()]
leftover = (~np.isclose(kept_vendor["adj_close"] / kept_vendor["close"], kept_vendor["ref_factor"], rtol=ADJ_RTOL)).sum()
assert leftover == 0, "inconsistent vendor adj_close survived the rebuild"

print(f"adj_close rebuilt {work['flag_adj_reconstructed'].sum()} "
      f"(of which vendor-inconsistent {work['flag_adj_vendor_inconsistent'].sum()}) | "
      f"quarantined {len(unresolved)} | "
      f"vendor rows kept and consistent with the reference: {len(kept_vendor)}")

work = work.drop(columns="ref_factor").reset_index(drop=True)
assert_price_integrity(work, "after adj_close rebuild")
assert np.isfinite(work["adj_close"]).all() and (work["adj_close"] > 0).all()

## Reconstruct vendor adj_close that disagrees with close x same-date reference factor
## (reconstruct_inconsistent=True from here on)

# Incremental and idempotent: it only touches untouched vendor rows that contradict the reference,
# so it gives the same result after the original rule and after the variant (where it does nothing).
if "flag_adj_vendor_inconsistent" not in work.columns:
    work["flag_adj_vendor_inconsistent"] = False  # created before 1.4, so FLAG_COLS and the saved table include it

# Same-date reference factor per row; NaN where the reference has no trusted row for that key.
ref_factor = pd.Series(
    work[KEY].merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")["ref_factor"].to_numpy(),
    index=work.index)
has_factor = ref_factor.notna()  # without a factor a vendor value cannot be checked, so it is kept

# Rows already rebuilt equal close x factor by construction, so only untouched vendor rows are tested.
agrees = pd.Series(np.isclose(work["adj_close"] / work["close"], ref_factor, rtol=ADJ_RTOL), index=work.index)
inconsistent = has_factor & ~work["flag_adj_reconstructed"] & ~agrees

work.loc[inconsistent, "adj_close"] = work["close"] * ref_factor
work.loc[inconsistent, "flag_adj_reconstructed"] = True  # 1.4's "no silent repairs: adj_close" check relies on this
work["flag_adj_vendor_inconsistent"] = work["flag_adj_vendor_inconsistent"].astype(bool) | inconsistent  # subset of flag_adj_reconstructed

# Recheck after the repair: no kept vendor value may contradict the reference, and adj_close stays valid.
kept_vendor = ~work["flag_adj_reconstructed"] & has_factor
assert np.isclose(work.loc[kept_vendor, "adj_close"] / work.loc[kept_vendor, "close"],
                  ref_factor[kept_vendor], rtol=ADJ_RTOL).all(), "inconsistent vendor adj_close survived"
assert np.isfinite(work["adj_close"]).all() and (work["adj_close"] > 0).all()
assert_price_integrity(work, "after reconstructing inconsistent adj_close")

print(f"rebuilt {inconsistent.sum()} inconsistent vendor adj_close values | "
      f"total adj_close rebuilt {work['flag_adj_reconstructed'].sum()} | "
      f"vendor rows kept and consistent: {kept_vendor.sum()}")

# Row accounting: every raw row is in exactly one place.
removed, quarantined = collect(removed_log), collect(quarantine_log)
ids = pd.concat([work["raw_row_id"], removed["raw_row_id"], quarantined["raw_row_id"]])
assert ids.is_unique and len(ids) == N_RAW, "row accounting does not reconcile"
print(f"raw {N_RAW} = retained {len(work)} + removed {len(removed)} + quarantined {len(quarantined)}")
display(removed["reason"].value_counts().to_frame("removed"))
display(quarantined["reason"].value_counts().to_frame("quarantined"))

In [ ]:
##Test cell: AAPL adj_close, original rule vs variant
import matplotlib.pyplot as plt

TICKER, START, END = "AAPL", "2024-01-01", "2024-12-31"

# Same snapshot, same function; only the switch differs.
orig, _, _ = rebuild_adj_close(work_pre_adj, ref, reconstruct_inconsistent=False)
new, _, _ = rebuild_adj_close(work_pre_adj, ref, reconstruct_inconsistent=True)


def window(frame: pd.DataFrame) -> pd.DataFrame:
    """One ticker, date window (ts is an ISO string, so string comparison is date order), indexed by date."""
    s = frame[(frame["ticker"] == TICKER) & frame["ts"].between(START, END)]
    return s.assign(ts=pd.to_datetime(s["ts"])).set_index("ts").sort_index()


vendor, orig, new, reference = window(work_pre_adj), window(orig), window(new), window(ref)
assert not new.empty, f"no {TICKER} rows in {START}..{END}"

# Dates where only the variant rebuilt the value; marked on both panels to show what changed.
only_variant = new.index[new["flag_adj_vendor_inconsistent"]]
print(f"{TICKER}: {len(new)} rows | {len(only_variant)} rebuilt only by the variant")

# sharey=False: each panel autoscales to its own data, so a small gap is not flattened by the other panel.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=False)

panels = [("Original rule", orig, "tab:blue"),
          ("Variant (also rebuild inconsistent vendor values)", new, "tab:red")]
for ax, (title, result, color) in zip(axes, panels):
    ax.plot(vendor.index, vendor["adj_close"], color="0.75", lw=3, label="vendor adj_close (before rebuild)")
    ax.plot(result.index, result["adj_close"], color=color, lw=1.3, label="adj_close after rule")
    ax.plot(reference.index, reference["ref_adj_close"], color="k", lw=0.9, ls=":", label="ohlc reference adj_close")
    ax.scatter(only_variant, result["adj_close"].reindex(only_variant), s=22, color="gold",
               edgecolor="k", zorder=3, label="rows the variant rebuilds")

    # Scale from the rule's output and the reference only, so one extreme vendor value
    # cannot crush the axis. Vendor points outside this range are clipped.
    shown = pd.concat([result["adj_close"], reference["ref_adj_close"]]).dropna()
    pad = 0.05 * (shown.max() - shown.min() or shown.max())
    ax.set_ylim(shown.min() - pad, shown.max() + pad)

    ax.set_title(title)
    ax.set_ylabel("adj_close")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)

fig.suptitle(f"{TICKER} adjusted close, {START} to {END}")
plt.tight_layout()
plt.show()

In [ ]:
# Row accounting: every raw row is in exactly one place.
removed, quarantined = collect(removed_log), collect(quarantine_log)
ids = pd.concat([work["raw_row_id"], removed["raw_row_id"], quarantined["raw_row_id"]])
assert ids.is_unique and len(ids) == N_RAW, "row accounting does not reconcile"
print(f"raw {N_RAW} = retained {len(work)} + removed {len(removed)} + quarantined {len(quarantined)}")
display(removed["reason"].value_counts().to_frame("removed"))
display(quarantined["reason"].value_counts().to_frame("quarantined"))

In [ ]:
## Final validation

ISO_DATE = r"\d{4}-\d{2}-\d{2}"
FLAG_COLS = [c for c in work.columns if c.startswith("flag_")]


def run_final_checks(clean, raw, removed, quarantined, ref) -> pd.DataFrame:
    """Independent recheck of the cleaned table: one row per check with its failure count.

    * Bounds are rewritten explicitly instead of reusing `ohlc_violations`, so a bug in
      the repair helpers cannot hide itself.
    * "No silent repairs": a row with no repair flag must equal its raw row.
    * Adjusted-price consistency: adj_close / close is finite and positive on every row, and
      every rebuilt adj_close equals close x the SAME-DATE reference factor.
    """
    prices = clean[PRICE_COLS + ["adj_close"]]
    o, h, l, c = (clean[k] for k in PRICE_COLS)
    factor = clean["adj_close"] / c
    ticker = clean["ticker"].astype(str)

    # Same-date reference factor for each retained row (ref has one row per key, order is kept).
    ref_factor = pd.Series(
        clean[KEY].merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")["ref_factor"].to_numpy(),
        index=clean.index)
    factor_matches = pd.Series(np.isclose(factor, ref_factor, rtol=1e-9), index=clean.index)  # False if no ref

    # Raw values of the same rows; NaN != NaN is True, so a raw NaN left unflagged is caught too.
    cmp = clean.merge(raw[["raw_row_id"] + VALUE_COLS], on="raw_row_id", suffixes=("", "_raw"))
    changed = lambda cols: pd.DataFrame({k: cmp[k].ne(cmp[f"{k}_raw"]) for k in cols}).any(axis=1)
    price_flagged = cmp[["flag_price_filled", "flag_ohlc_repaired", "flag_extreme_return_interpolated"]].any(axis=1)

    ids = pd.concat([clean["raw_row_id"], removed["raw_row_id"], quarantined["raw_row_id"]])

    checks = {
        "unique (ticker, ts) keys": clean.duplicated(KEY).sum(),
        "ticker label clean (no markup/whitespace, upper-case)":
            (ticker.str.contains(r"[<>\s]") | ticker.ne(ticker.str.upper())).sum(),
        "ts is an ISO date YYYY-MM-DD":
            (~clean["ts"].astype(str).str.fullmatch(ISO_DATE)
             | pd.to_datetime(clean["ts"], format="%Y-%m-%d", errors="coerce").isna()).sum(),
        "no missing / non-finite / non-positive raw or adjusted price": (~np.isfinite(prices) | prices.le(0)).sum().sum(),
        "finite non-negative volume": (~np.isfinite(clean["volume"]) | clean["volume"].lt(0)).sum(),
        "OHLC bounds: low <= open, close <= high": ((h < o) | (h < c) | (h < l) | (l > o) | (l > c)).sum(),
        "adj_close / close finite and positive": (~np.isfinite(factor) | factor.le(0)).sum(),
        "rebuilt adj_close = close x same-date reference factor":
            (clean["flag_adj_reconstructed"] & ~factor_matches).sum(),
        "every repaired close has a rebuilt adj_close":
            (clean["flag_close_repaired"] & ~clean["flag_adj_reconstructed"]).sum(),
        "no silent repairs: open/high/low/close": (changed(PRICE_COLS) & ~price_flagged).sum(),
        "no silent repairs: adj_close": (changed(["adj_close"]) & ~cmp["flag_adj_reconstructed"]).sum(),
        "no silent repairs: volume": (changed(["volume"]) & ~cmp["flag_volume_filled"]).sum(),
        "row accounting: every raw row in exactly one table":
            ids.duplicated().sum() + abs(len(ids) - len(raw)) + (~ids.isin(raw["raw_row_id"])).sum(),
        "removed / quarantined rows all carry a reason":
            removed["reason"].isna().sum() + quarantined["reason"].isna().sum(),
    }
    out = pd.DataFrame({"check": list(checks), "n_failures": [int(v) for v in checks.values()]})
    return out.assign(passed=out["n_failures"].eq(0))


removed, quarantined = collect(removed_log), collect(quarantine_log)
final_checks = run_final_checks(work, df, removed, quarantined, ref)
display(final_checks)
assert final_checks["passed"].all(), "final validation failed: see the table above"

# Informational, not a failure: untouched vendor adj_close values that disagree with the reference
# factor on the same date. They were reported (not altered) in the adj_close step.
vendor = work.merge(ref[KEY + ["ref_factor"]], on=KEY, how="left")
vendor = vendor[~vendor["flag_adj_reconstructed"] & vendor["ref_factor"].notna()]
n_disagree = (~np.isclose(vendor["adj_close"] / vendor["close"], vendor["ref_factor"], rtol=ADJ_RTOL)).sum()
print(f"All {len(final_checks)} checks passed on {len(work)} rows. "
      f"Vendor adj_close disagreeing with the reference factor: {n_disagree} of {len(vendor)} rows.")

In [ ]:
## Before / after statistics

def profile_stats(rows: pd.DataFrame) -> pd.Series:
    """Headline data quality metrics for one version of the table.

    * Row and ticker counts use every row. All other counts use distinct rows with a usable
      key, the same basis as the Data Quality Report in 1.2, so 'before' ties back to it.
    """
    distinct = rows[rows["key_issue"].isna()].drop_duplicates(KEY + VALUE_COLS)
    cells = distinct[PRICE_COLS + ["adj_close"]]
    high_bad, low_bad = ohlc_violations(distinct)
    stats = {
        "rows": len(rows),
        "unique tickers (normalized)": rows["ticker"].nunique(),
        "exact duplicate rows": rows.duplicated(KEY + VALUE_COLS).sum(),
        "extra rows on a repeated (ticker, ts) key": distinct.duplicated(KEY).sum(),
        **{f"missing {c}": distinct[c].isna().sum() for c in VALUE_COLS},
        "non-positive / infinite price cells": (cells.le(0) | np.isinf(cells)).sum().sum(),
        "impossible OHLC rows": (high_bad | low_bad).sum(),
        "zero-volume rows": distinct["volume"].eq(0).sum(),
        "extreme-return days (3 std)": len(check_extreme_returns(distinct)),
    }
    return pd.Series(stats).astype(int)


print(f"Ticker labels: {df['raw_ticker'].nunique()} raw -> {work['ticker'].nunique()} normalized (retained rows)")

before_after = pd.concat({"before": profile_stats(df), "after": profile_stats(work)}, axis=1)
before_after["change"] = before_after["after"] - before_after["before"]
display(before_after)
# 'after' still shows extreme-return days: moves confirmed by the reference or split-like are kept
# and flagged, and the z-scores are recomputed on the repaired series.

# Where every raw row went.
accounting = pd.concat([
    pd.DataFrame({"table": "retained (ohlc_hw_cleaned)", "reason": "-", "n_rows": [len(work)]}),
    removed.groupby("reason").size().rename("n_rows").reset_index().assign(table="removed (ohlc_hw_removed)"),
    quarantined.groupby("reason").size().rename("n_rows").reset_index().assign(table="quarantined (ohlc_hw_quarantine)"),
], ignore_index=True)[["table", "reason", "n_rows"]]
accounting["pct_of_raw"] = (accounting["n_rows"] / N_RAW * 100).round(2)
display(accounting)
print(f"raw {N_RAW} = retained {len(work)} + removed {len(removed)} + quarantined {len(quarantined)}")

# How much of the retained data was repaired, and how the kept extreme returns were classified.
flag_summary = work[FLAG_COLS].sum().to_frame("n_rows")
flag_summary["pct_of_retained"] = (flag_summary["n_rows"] / len(work) * 100).round(2)
display(flag_summary)
display(work["extreme_return_class"].value_counts(dropna=False).to_frame("n_rows"))

# Quarantine concentrated in a few tickers usually points at a vendor problem worth reviewing.
display(quarantined.groupby("ticker").size().sort_values(ascending=False).head(10).to_frame("n_quarantined"))

# Distribution shift: repairs should not move the overall shape of the data.
display(pd.concat({"before": df[VALUE_COLS].describe().T, "after": work[VALUE_COLS].describe().T}, axis=1).round(2))

In [ ]:
## Write outputs to data.db

from contextlib import closing

DB_PATH = "data/data.db"
CLEAN_COLS = (["raw_row_id", "ticker", "ts"] + VALUE_COLS + ["raw_ticker", "raw_ts"]
              + FLAG_COLS + ["extreme_return_class"])


def for_sqlite(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert to plain dtypes SQLite stores predictably.

    * bool -> 0/1, timestamps -> ISO date text, pandas string -> object, missing -> NULL.
    """
    out = frame.copy()
    for col in out.columns:
        s = out[col]
        if pd.api.types.is_bool_dtype(s):
            out[col] = s.astype(int)
        elif pd.api.types.is_datetime64_any_dtype(s):
            out[col] = s.dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_string_dtype(s):
            out[col] = s.astype(object).where(s.notna(), None)
    return out


outputs = {
    "ohlc_hw_cleaned": work[CLEAN_COLS],
    "ohlc_hw_quarantine": quarantined,   # LOG_COLS + reason + note (stated uncertainty)
    "ohlc_hw_removed": removed,           # LOG_COLS + reason
    "ohlc_hw_ticker_audit": ticker_audit,
    "ohlc_hw_dq_report": dq_report.assign(severity=dq_report["severity"].astype(str)),
}
# Guard: the raw table and the reference table are never written to.
assert not {"ohlc_hw", "ohlc"} & set(outputs)

# One transaction: either every output table is replaced or none is.
with closing(sqlite3.connect(DB_PATH)) as conn, conn:
    for name, frame in outputs.items():
        for_sqlite(frame).to_sql(name, conn, if_exists="replace", index=False)

# Read back from the database, not from memory, to verify what was actually saved.
with closing(sqlite3.connect(DB_PATH)) as conn:
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in [*outputs, "ohlc_hw"]}
    n_bad_ts = conn.execute(
        "SELECT COUNT(*) FROM ohlc_hw_cleaned "
        "WHERE ts NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'").fetchone()[0]
    n_dup_keys = conn.execute(
        "SELECT COUNT(*) FROM (SELECT 1 FROM ohlc_hw_cleaned GROUP BY ticker, ts HAVING COUNT(*) > 1)").fetchone()[0]
    n_bad_rows = conn.execute("""
        SELECT COUNT(*) FROM ohlc_hw_cleaned
        WHERE open IS NULL OR high IS NULL OR low IS NULL OR close IS NULL OR adj_close IS NULL OR volume IS NULL
           OR open <= 0 OR high <= 0 OR low <= 0 OR close <= 0 OR adj_close <= 0 OR volume < 0
           OR high < MAX(open, close, low) OR low > MIN(open, close, high)""").fetchone()[0]
    ts_range = conn.execute("SELECT MIN(ts), MAX(ts) FROM ohlc_hw_cleaned").fetchone()

display(pd.Series(counts, name="rows in database").to_frame())
assert counts["ohlc_hw_cleaned"] == len(work)
assert counts["ohlc_hw_removed"] == len(removed) and counts["ohlc_hw_quarantine"] == len(quarantined)
assert counts["ohlc_hw_cleaned"] + counts["ohlc_hw_removed"] + counts["ohlc_hw_quarantine"] == N_RAW
assert counts["ohlc_hw"] == N_RAW, "raw table row count changed"
assert n_bad_ts == 0 and n_dup_keys == 0 and n_bad_rows == 0
print(f"Saved and verified. ts range {ts_range[0]} to {ts_range[1]}; "
      f"{counts['ohlc_hw_cleaned']} cleaned + {counts['ohlc_hw_removed']} removed + "
      f"{counts['ohlc_hw_quarantine']} quarantined = {N_RAW} raw rows.")